# Generating Input for MODFLOW

This notebook shows how to convert hydraulic soil properties from `pedon` models into parameters that can be used in FloPy-based MODFLOW workflows.

The workflow is:
1. Build or load a soil hydraulic model in `pedon`.
2. Evaluate water retention and conductivity over a pressure-head range.
3. Fit a MODFLOW-oriented parameterization (here: `Panday`).
4. Create MODFLOW-USG / MODFLOW 6 package inputs.

## FloPy references
- Bakker, M., Post, V., Langevin, C. D., Hughes, J. D., White, J. T., Starn, J. J., & Fienen, M. N. (2016). *Scripting MODFLOW Model Development Using Python and FloPy*. Groundwater, 54(5), 733-739. https://doi.org/10.1111/gwat.12413/epdf
- Hughes, J. D., Langevin, C. D., Paulinski, S. R., Larsen, J. D., & Brakenhoff, D. (2024). *FloPy Workflows for Creating Structured and Unstructured MODFLOW Models*. Groundwater, 62, 124-139. https://doi.org/10.1111/gwat.13327

In [ ]:
import flopy as fp
import numpy as np
import pandas as pd

import pedon as pe
import matplotlib.pyplot as plt

## Converting soil models

Here we generate synthetic retention and conductivity data from a `Genuchten` soil model and fit a `Panday` model to those data. This gives us a practical bridge from commonly available soil descriptions to parameters needed by MODFLOW packages.

This routine is described in more detail in the curve_fitting notebook.

In [ ]:
gen = pe.Soil("Sand").from_name(pe.Genuchten, source="HYDRUS").model
# gen = pe.Genuchten(k_s=100.0, theta_s=0.4, theta_r=0.05, alpha=0.1, n=2.0)

ax = gen.plot()

# sample the SWRC and HCF at 11 points between 10^-4 and 10^6 cm of pressure head
h = np.logspace(-4, 6, num=11)
theta = gen.theta(h)
k = gen.k(h)

ax.scatter(theta, h, color="k", label="Sample for fitting")
ax.legend()

### Fit SWRC and HCF

In [ ]:
pbounds = pe.get_params(pe.Panday)
pbounds

pan_both = pe.SoilSample(
    theta=theta,
    k=k,
    h=h).fit(pe.Panday, pbounds=pbounds)
pan_both

### Fit only the HCF

In [ ]:
# Note that we set the bounds for theta_r, theta_s, alpha and n very small.
# This is because we want to fix these parameters during optimization and only
# optimize for the relative hydraulic conductivity curve with the brook parameter.

pbounds_brook = pbounds.update(
    pd.DataFrame(
    data=[
        [
            gen.theta_r,
            gen.theta_r - 1e-10,
            gen.theta_r + 1e-10,
        ],
        [
            gen.theta_s,
            gen.theta_s - 1e-10,
            gen.theta_s + 1e-10,
        ],
        [
            gen.alpha,
            gen.alpha - 1e-10,
            gen.alpha + 1e-10,
        ],
        [gen.n, gen.n - 1e-10, gen.n + 1e-10],
    ],
    index=["theta_r", "theta_s", "alpha", "beta"],
    columns=["p_ini", "p_min", "p_max"],
))

pan = pe.SoilSample(h=h, theta=theta, k=k).fit(pe.Panday, pbounds=pbounds, k_s=gen.k_s)
pan

In [ ]:
f, axd = plt.subplot_mosaic([["swrc", "hcf"]], sharey=True)
pe.plot_swrc(gen, ax=axd["swrc"], label="fitted")


## MODFLOW-USG Transport Parameters

The fitted `Panday` object (`pan`) can be mapped directly to `MfUsgLpf` soil-related inputs. Please make sur that the units are consistent with your MODFLOW model. 

In [ ]:
fp.mfusg.MfUsgLpf(
    ...,
    hk=pan.k_s / 100.0, # convert from cm/day to m/day
    sy=pan.theta_s,
    ss=pan.ss,
    alpha=pan.alpha * 100, # convert from 1/cm to 1/m
    beta=pan.beta,
    sr=pan.sr,
    brook=pan.brook,
)

## MODFLOW 6 UZF

For UZF, we provide effective unsaturated-zone parameters through `packagedata` (for example `vks`, `thtr`, `thts`, and `eps`). Unlike full retention-curve formulations, UZF uses a reduced parameterization of unsaturated flow behavior. In this example, `pedon`-derived parameters are used to populate those terms in a consistent way.

In [ ]:
# packagedata : [(ifno, cellid, landflag, ivertcon, surfdep, vks, thtr, thts, thti, eps, boundname)]
packagedata = [
    (..., ..., ..., ..., ..., pan.k_s / 100.0, pan.theta_r, pan.theta_s, ..., pan.brook)
]
fp.mf6.ModflowGwfuzf(
    ...,
    packagedata=packagedata,
)